# Causal Forest multi-output — fit and save

This notebook trains per-endpoint causal forest models (an alternative to the T-learner in
`04_fit_calibrated_models.ipynb`). It fits **three explicit flexibility levels** of the base
model — `regularized` (shallow, heavily regularized), `medium`, and `flexible` (deep, high
capacity) — defined in `casual_multioutput_pipeline.CF_MODEL_PRESETS`. Each preset scales both
the causal-forest hyper-parameters and the first-stage LGBM nuisances. For each level it:

1. trains a multi-output `CausalMultiOutputPipeline` on the full data;
2. **saves** it to `models/CausalForest/CausalForest_multioutput_<level>.joblib` (the `medium`
   model is also saved as `CausalForest_multioutput.joblib`, the default 07 loads).

The three levels double as a sensitivity analysis: if no CATE heterogeneity survives even the
`flexible` model — while `regularized` collapses toward the ATE — the homogeneous-effect
conclusion is robust to model capacity. Optuna tuning is retained behind `RUN_OPTUNA` for
reference but no longer drives the saved models.

The first-stage (nuisance) learners `model_y=E[Y|x]` and `model_t=E[T|x]` travel with each saved
object, so the analysis notebook reuses the exact same configuration instead of re-hardcoding it.

The CATE estimation, its visualisations and the predictive evaluation (out-of-fold ROC,
held-out confusion matrices, heterogeneity tests) live in `07_causal_forest_analysis.ipynb`.

In [46]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna

from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_curve, roc_auc_score
from econml.dml import CausalForestDML
from lightgbm import LGBMRegressor, LGBMClassifier
from IPython.display import display

from thesis_utils import predict_proba_matrix, plot_confusion_matrices

DATA_DIR = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'data'))
MODELS_DIR = os.path.normpath(os.path.join(os.getcwd(), '..', 'models'))

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

## Configuration

`USE_KNN_IMPUTER` controls imputation strategy. The saved models come from the three explicit
`CF_MODEL_PRESETS`; `DEFAULT_LEVEL` is the one mirrored to the canonical filename. Optuna tuning
is optional and gated by `RUN_OPTUNA` (`TUNE_CV` folds, `N_TRIALS` trials, overridable via
`CATE_N_TRIALS`).

In [47]:
USE_KNN_IMPUTER = True
KNN_NEIGHBORS = 5

RUN_OPTUNA = True                   # the three explicit presets drive the saved models;
                                    # flip on to explore tuned hyper-parameters for reference
TUNE_CV = 5                         # CV folds for the Optuna objective (5 -> more stable estimate)
N_TRIALS = int(os.environ.get('CATE_N_TRIALS', 2000))   # Optuna trials (keep modest: a huge
                                    # budget overfits the noisy targeting metric = winner's curse)

DEFAULT_LEVEL = 'flexible'            # preset also saved as the canonical default model

CALIBRATION_METHOD = 'sigmoid'      # Only for reference; CausalForest does not use it

print(f'Imputation strategy        : {"KNN" if USE_KNN_IMPUTER else "Median"}')
print(f'Optuna tuning              : {"on" if RUN_OPTUNA else "off (using presets)"}')
print(f'Optuna trials              : {N_TRIALS}')
print(f'CV folds for tuning        : {TUNE_CV}')

Imputation strategy        : KNN
Optuna tuning              : on
Optuna trials              : 2000
CV folds for tuning        : 5


## Load data and split by treatment arm

Load the cleaned features and 5 endpoints, and encode treatment as 0 (abbreviated DAPT) /
1 (prolonged DAPT).

In [48]:
X = pd.read_parquet(os.path.join(DATA_DIR, 'X_features.parquet'))
y_df = pd.read_parquet(os.path.join(DATA_DIR, 'y_targets.parquet'))

TARGETS = [
    'cec_barc235_335d',
    'cec_cvdeath_335d',
    'cec_mi_335d',
    'cec_stroke_335d',
    'cec_bleed_335d',
]
TARGET_LABELS = ['barc_235', 'death', 'mi', 'stroke', 'bleed']
y_df = y_df[TARGETS]

X_num = X.select_dtypes(include=[np.number])

regimen = X['regimen']
T = regimen.map({'prolonged DAPT': 1, 'abbreviated DAPT': 0}).to_numpy()

print(f'Samples: {len(X_num)}, Features: {X_num.shape[1]}')
print(f'Treatment: {np.sum(T)} prolonged DAPT, {len(T) - np.sum(T)} abbreviated DAPT')
print(f'Targets: {len(TARGETS)} endpoints')
print(f'Event rates: {dict(y_df.mean().round(3))}')

Samples: 4579, Features: 63
Treatment: 2284 prolonged DAPT, 2295 abbreviated DAPT
Targets: 5 endpoints
Event rates: {'cec_barc235_335d': np.float64(0.078), 'cec_cvdeath_335d': np.float64(0.018), 'cec_mi_335d': np.float64(0.024), 'cec_stroke_335d': np.float64(0.008), 'cec_bleed_335d': np.float64(0.111)}


In [49]:
# Dump the prepared arrays for the standalone parallel tuner (run_optuna_tuning.py),
# so it can reproduce the objective without re-running this notebook's preprocessing.
np.savez(
    os.path.join(os.getcwd(), 'tuning_data.npz'),
    X=X_num.values, Y=y_df.values, T=np.asarray(T).ravel(),
    use_knn=USE_KNN_IMPUTER, knn_k=KNN_NEIGHBORS, tune_cv=TUNE_CV,
)
print('Wrote tuning_data.npz for run_optuna_tuning.py')

Wrote tuning_data.npz for run_optuna_tuning.py


## Custom Causal Multi-output Pipeline

Wrapper around per-endpoint CausalForestDML with robust imputation and preprocessing.
Trains one causal forest per target; `predict_cate` returns shape `(n_samples, n_targets)`.

In [50]:
from casual_multioutput_pipeline import (
    CausalMultiOutputPipeline,
    CF_MODEL_PRESETS,
    make_preset_pipeline,
)

print('Base-model flexibility presets:')
for level, preset in CF_MODEL_PRESETS.items():
    print(f"  {level:12s} forest={preset['cf_params']}")

Base-model flexibility presets:
  regularized  forest={'n_estimators': 400, 'max_depth': 2, 'min_samples_leaf': 40, 'max_samples': 0.3, 'min_balancedness_tol': 0.3, 'min_impurity_decrease': 0.001}
  medium       forest={'n_estimators': 500, 'max_depth': 4, 'min_samples_leaf': 15, 'max_samples': 0.4}
  flexible     forest={'n_estimators': 600, 'max_depth': 8, 'min_samples_leaf': 5, 'max_samples': 0.5}


## Hyper-parameter tuning with Optuna (parallel, standalone script)

Reference only — the saved models come from the three presets, not from tuning. Tuning runs as a
**process-based** parallel script (`run_optuna_tuning.py`): many single-threaded worker processes
share one Optuna study on a file journal, so it scales across all cores without the GIL or the
nested-thread oversubscription of `study.optimize(n_jobs=...)`.

The objective ranks patients by predicted CATE and maximises the **AUTOC** (Area Under the TOC
Curve) **for the bleeding endpoint** — the mean doubly-robust "gain over random" integrated across
*all* treatment fractions (econml's uplift coefficient, via `DRTester`), not just the extreme top
group. It scores that **single** endpoint (`target_idx=BLEED_IDX`) instead of averaging the AUTOC
across all five targets, so it selects the config whose CATE ranking best targets **bleeding
risk**. It is optimised as a **lower confidence bound** (`AUTOC − z·se`). Using the lower bound —
not the raw estimate — and scoring collapsed (constant) CATEs as 0 makes the search **robust to
winner's curse**: with weak real heterogeneity in this cohort, chasing the raw metric over
thousands of trials just fits noise and collapses the forest to a constant CATE.

The search tunes both the causal forest and, via a single `nuisance_level` categorical, the
**first-stage nuisance flexibility** (`regularized`/`medium`/`flexible` presets for `E[Y|x]` /
`E[T|x]`): better-estimated nuisances give less-noisy doubly-robust pseudo-outcomes and hence a
tighter AUTOC lower bound.

Run it from a terminal (after executing the `tuning_data.npz` dump cell above):

```bash
cd Meta-learning/causal_forest
python run_optuna_tuning.py --workers 64 --trials 200
```

The study is resumable (re-running adds trials to the same journal). The cell below loads the
finished study to inspect the best hyper-parameters.

In [51]:
# Parallel Optuna tuning runs in run_optuna_tuning.py (see the markdown above).
# Here we just load a finished study from its journal to read the best params.
import time
import shutil
import tempfile
from optuna.storages import JournalStorage, InMemoryStorage
from optuna.storages.journal import JournalFileBackend

# --- pick which tuning run to read --------------------------------------------
# Each journal holds exactly ONE study, so ONLY the FILE has to be set here; the
# study name is auto-detected (STUDY_NAME=None). Switching runs = change STORAGE_PATH
# only, and no more KeyError from a stale/typo'd name. Journals in this folder:
#   optuna_cate_top5_bleed.journal  -> 'cate_top5_bleed'   (top-5% targeting gain, bleed only; what run_optuna_tuning.py writes NOW)
#   optuna_cate_autoc_bleed.journal -> 'cate_autoc_bleed'  (AUTOC lower bound, bleed only)
#   optuna_cate_top5_reg.journal    -> 'cate_top5_lb_reg'  (older top-5% study, forest-only search space)
#   optuna_cate_autoc_nuis.journal  -> 'cate_autoc_nuis'   (older multi-target AUTOC study)
STORAGE_PATH = os.path.join(os.getcwd(), 'optuna_cate_autoc_bleed.journal')
STUDY_NAME = None                 # None -> auto-detect the single study in STORAGE_PATH


def load_optuna_study(storage_path, study_name=None, retries=6, pause=0.7):
    """Robustly load an Optuna study from a JournalFileBackend.

    * study_name=None auto-resolves when the journal holds exactly one study, so
      switching journals only needs STORAGE_PATH changed (nothing to keep in sync).
    * Reads a *static snapshot copy* (nothing writes to the copy) and retries on a
      torn snapshot, so a load that races a still-running run_optuna_tuning.py
      recovers instead of raising 'Record does not exist.'.
    * Materialises the study into an in-memory storage before returning, so the
      returned object does NOT lazily re-read the on-disk journal. A JournalStorage
      only holds the *path*; without this copy, `study.best_params` would re-open the
      temp snapshot AFTER the finally-block deleted it -> FileNotFoundError.
    """
    last_err = None
    for _ in range(retries):
        tmp = os.path.join(tempfile.gettempdir(), f'_snap_{os.path.basename(storage_path)}')
        try:
            shutil.copy2(storage_path, tmp)                 # frozen snapshot: no writer races our read
            storage = JournalStorage(JournalFileBackend(tmp))
            names = [s.study_name for s in storage.get_all_studies()]
            name = study_name or (names[0] if len(names) == 1 else None)
            if name is None:
                raise ValueError(f'{os.path.basename(storage_path)} holds studies {names}; '
                                 'set STUDY_NAME to pick one.')
            if name not in names:
                raise KeyError(f'study {name!r} not in {os.path.basename(storage_path)}; '
                               f'available: {names}.')
            # Pull every trial into memory now, while the snapshot still exists, so the
            # returned study is fully detached from `tmp` (which finally deletes below).
            mem = InMemoryStorage()
            optuna.copy_study(from_study_name=name, from_storage=storage, to_storage=mem)
            return optuna.load_study(study_name=name, storage=mem)
        except (ValueError, KeyError):
            raise                                           # wrong file/name: a real config error, surface it
        except Exception as e:
            last_err = e                                    # torn snapshot mid-append: copy again
            time.sleep(pause)
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)
    raise RuntimeError(
        f'Could not read {os.path.basename(storage_path)} after {retries} tries — the tuner is '
        f'probably still writing it; wait for run_optuna_tuning.py to finish and re-run. '
        f'Last error: {type(last_err).__name__}: {last_err}')


best_params, best_value = None, None
if RUN_OPTUNA and os.path.exists(STORAGE_PATH):
    study = load_optuna_study(STORAGE_PATH, STUDY_NAME)
    best_params, best_value = study.best_params, study.best_value
    print(f'Loaded study "{study.study_name}" from {os.path.basename(STORAGE_PATH)} '
          f'— {len(study.trials)} trials')
    print(f'Best CV score (−targeting-gain lower bound): {best_value:.4f}')
    print(f'Best params  : {best_params}')
else:
    print(f'No tuning study at {os.path.basename(STORAGE_PATH)} — run '
          '`python run_optuna_tuning.py` first (or leave RUN_OPTUNA off to use the presets).')

Loaded study "cate_autoc_bleed" from optuna_cate_autoc_bleed.journal — 23361 trials
Best CV score (−targeting-gain lower bound): -0.0143
Best params  : {'subforest_size': 49, 'n_subforests': 73, 'max_depth': 10, 'min_samples_leaf': 54, 'min_samples_split': 93, 'max_samples': 0.4087611569036036, 'max_features': 1.0, 'min_balancedness_tol': 0.41757555974146765, 'min_impurity_decrease': 0.002801219942552527, 'nuisance_level': 'regularized'}


## Fit the three flexibility levels and save

Train one `regularized`, one `medium`, and one `flexible` causal forest on the full dataset with
inference (bootstrap CIs) enabled, and save each to its own file. The `DEFAULT_LEVEL` model is
also mirrored to `CausalForest_multioutput.joblib` so `07` keeps loading a canonical default.

In [52]:
SAVE_DIR = os.path.join(MODELS_DIR, 'CausalForest')
os.makedirs(SAVE_DIR, exist_ok=True)

"""
# Re-saving all levels: drop any stale causal-forest files first.
for stale in glob.glob(os.path.join(SAVE_DIR, 'CausalForest_*.joblib')):
    os.remove(stale)
"""

pipelines = {}
"""
for level in CF_MODEL_PRESETS:
    print(f'Fitting "{level}" causal forest ...')
    pipe = make_preset_pipeline(
        level, inference=True,
        use_knn_imputer=USE_KNN_IMPUTER, knn_neighbors=KNN_NEIGHBORS,
    )
    pipe.fit(X_num.values, y_df.values, T)
    pipelines[level] = pipe
    joblib.dump(pipe, os.path.join(SAVE_DIR, f'CausalForest_multioutput_{level}.joblib'))
    print(f'  saved -> models/CausalForest/CausalForest_multioutput_{level}.joblib')

# Mirror the default level to the canonical filename 07 loads.
joblib.dump(pipelines[DEFAULT_LEVEL], os.path.join(SAVE_DIR, 'CausalForest_multioutput.joblib'))
print(f'\nDefault ({DEFAULT_LEVEL}) mirrored -> models/CausalForest/CausalForest_multioutput.joblib')
print('Done. Run 07_causal_forest_analysis.ipynb to analyse the effects.')
"""

'\nfor level in CF_MODEL_PRESETS:\n    print(f\'Fitting "{level}" causal forest ...\')\n    pipe = make_preset_pipeline(\n        level, inference=True,\n        use_knn_imputer=USE_KNN_IMPUTER, knn_neighbors=KNN_NEIGHBORS,\n    )\n    pipe.fit(X_num.values, y_df.values, T)\n    pipelines[level] = pipe\n    joblib.dump(pipe, os.path.join(SAVE_DIR, f\'CausalForest_multioutput_{level}.joblib\'))\n    print(f\'  saved -> models/CausalForest/CausalForest_multioutput_{level}.joblib\')\n\n# Mirror the default level to the canonical filename 07 loads.\njoblib.dump(pipelines[DEFAULT_LEVEL], os.path.join(SAVE_DIR, \'CausalForest_multioutput.joblib\'))\nprint(f\'\nDefault ({DEFAULT_LEVEL}) mirrored -> models/CausalForest/CausalForest_multioutput.joblib\')\nprint(\'Done. Run 07_causal_forest_analysis.ipynb to analyse the effects.\')\n'

## Fit and save the Optuna-tuned model

If a tuning study was run (`run_optuna_tuning.py` → `best_params` loaded above), refit a causal
forest with those hyper-parameters on the full data and save it as
`CausalForest_multioutput_tuned.joblib`. Set `MODEL_LEVEL = 'tuned'` in
`07_causal_forest_analysis.ipynb` to visualise it. Optuna searched `subforest_size` and
`n_subforests`, so `n_estimators` is reconstructed here as their product (same as the objective).

In [53]:
SAVE_DIR = os.path.join(MODELS_DIR, 'CausalForest')
os.makedirs(SAVE_DIR, exist_ok=True)

if best_params:
    # Rebuild cf_params exactly as run_optuna_tuning.objective did (n_estimators is the
    # product of the two searched sub-forest parameters); enable inference like the presets.
    cf_params_tuned = {
        'n_estimators':          best_params['n_subforests'] * best_params['subforest_size'],
        'subforest_size':        best_params['subforest_size'],
        'max_depth':             best_params['max_depth'],
        'min_samples_leaf':      best_params['min_samples_leaf'],
        'max_samples':           best_params['max_samples'],
        'max_features':          best_params['max_features'],
         #'min_impurity_decrease': best_params['min_impurity_decrease'],
        'inference':             True,
    }
    # The tuner also picks a first-stage nuisance flexibility level; rebuild the same
    # nuisance_params so the saved model matches what was scored (.get keeps older,
    # forest-only studies loadable -> default 'medium').
    nuisance_level_tuned = best_params.get('nuisance_level', 'medium')
    nuisance_params_tuned = dict(CF_MODEL_PRESETS[nuisance_level_tuned]['nuisance_params'])
    print(f'Fitting Optuna-tuned causal forest — nuisance_level={nuisance_level_tuned!r}, '
          f'cf_params={cf_params_tuned}')
    tuned_pipe = CausalMultiOutputPipeline(
        use_knn_imputer=USE_KNN_IMPUTER, knn_neighbors=KNN_NEIGHBORS,
        cf_params=cf_params_tuned,
        nuisance_params=nuisance_params_tuned,
    )
    tuned_pipe.fit(X_num.values, y_df.values, T)
    joblib.dump(tuned_pipe, os.path.join(SAVE_DIR, 'CausalForest_multioutput_tuned.joblib'))
    print('  saved -> models/CausalForest/CausalForest_multioutput_tuned.joblib')

    # Verdict: did the (robust) tuning find a reliable AUTOC (targeting signal), or just noise?
    #   - collapsed  -> the tuned forest is ~constant CATE (degenerate, no ranking)
    #   - reliable   -> best_value is meaningfully negative, i.e. mean AUTOC lower bound > 0
    std_cate = tuned_pipe.predict_cate(X_num.values).std(axis=0)
    collapsed = float(np.max(std_cate)) < 1e-3
    reliable = (best_value is not None) and (best_value < -1e-3)
    print(f'\n  std(CATE) per target: {dict(zip(TARGET_LABELS, np.round(std_cate, 4)))}')
    if reliable and not collapsed:
        print("  VERDICT: PASS — reliable AUTOC (targeting gain across the curve); MODEL_LEVEL='tuned' in 07 is meaningful.")
    else:
        why = 'collapsed to a ~constant CATE' if collapsed else 'no reliable AUTOC (best_value ≈ 0)'
        print(f"  VERDICT: FAIL — {why}. Prefer a preset in 07 (MODEL_LEVEL='flexible').")
        print('           (Expected for this cohort: no significant CATE heterogeneity.)')
else:
    print('No best_params found — run run_optuna_tuning.py first, then re-run the '
          'tuning cell above to load the study. Tuned model not saved.')

Fitting Optuna-tuned causal forest — nuisance_level='regularized', cf_params={'n_estimators': 3577, 'subforest_size': 49, 'max_depth': 10, 'min_samples_leaf': 54, 'max_samples': 0.4087611569036036, 'max_features': 1.0, 'inference': True}
  saved -> models/CausalForest/CausalForest_multioutput_tuned.joblib

  std(CATE) per target: {'barc_235': np.float64(0.0041), 'death': np.float64(0.0035), 'mi': np.float64(0.0034), 'stroke': np.float64(0.0065), 'bleed': np.float64(0.0056)}
  VERDICT: PASS — reliable AUTOC (targeting gain across the curve); MODEL_LEVEL='tuned' in 07 is meaningful.


## Estimate CATE on full cohort — compare flexibility levels

Generate CATE predictions for all patients under each preset. The key diagnostic is `std_CATE`:
it should grow from `regularized` to `flexible`. If even the `flexible` model keeps it small,
the lack of heterogeneity is robust to model capacity rather than a regularization artefact.

In [54]:
summaries = {}
for level, pipe in pipelines.items():
    cate_preds = pipe.predict_cate(X_num.values)
    cate = pd.DataFrame(cate_preds, columns=TARGET_LABELS, index=X_num.index)
    summaries[level] = pd.DataFrame({
        'mean_CATE': cate.mean(),
        'std_CATE': cate.std(),
        'pct_lDAPT_better': (cate < 0).mean() * 100,
        'pct_sDAPT_better': (cate > 0).mean() * 100,
    })
    print(f'\nCATE summary — {level}')
    display(summaries[level].round(4))

# Side-by-side CATE dispersion across levels (heterogeneity signal).
std_by_level = pd.DataFrame({lvl: s['std_CATE'] for lvl, s in summaries.items()})
print('\nstd(CATE) by flexibility level')
display(std_by_level.round(4))


std(CATE) by flexibility level


""
